In [12]:
import pandas as pd
import numpy as np
import os
from sleep_research_functions import load_sleep_data, write_to_hdf5_file
pd.set_option("display.max_rows", 101)
from tqdm import tqdm

In [14]:
df = pd.DataFrame()

In [15]:
recordings_dir = '/media/mad3/Projects/AirGo_PSG/10Hz_data'
for study_id in tqdm(range(500)):
    try:
        filepath = os.path.join(recordings_dir, 'psg_airgo_10hz_'+str(study_id).zfill(3)+'.h5')
        if not os.path.exists(filepath): continue

        data, hdr = load_sleep_data(filepath)
        data.Apnea[(data.Apnea>0).diff()>0]=0
        data.Apnea[(data.Apnea>0).diff()<0]=0
        fs = hdr['fs']
        hours = sum(np.isin(data.Stage,[1,2,3,4]))/fs/3600


        # AHI
        df.loc[study_id, 'AHI'] = sum(np.diff(np.isin(data.Apnea, [1,2,3,4]))>0)/hours

        obs = data.Apnea==1
        obs = sum(np.diff(obs)>0)
        cen = data.Apnea==2
        cen = sum(np.diff(cen)>0)
        df.loc[study_id, 'OI'] = obs/hours
        df.loc[study_id, 'CI'] = cen/hours

        df.loc[study_id, 'Arousal_Index'] = len([x for x in data.Annotation if 'Arousal' in x])/hours

        df.loc[study_id, 'W'] = sum(np.isin(data.Stage, [5]))/data.shape[0]*100
        df.loc[study_id, 'R'] = sum(np.isin(data.Stage, [4]))/data.shape[0]*100
        df.loc[study_id, 'N1'] = sum(np.isin(data.Stage, [3]))/data.shape[0]*100
        df.loc[study_id, 'N2'] = sum(np.isin(data.Stage, [2]))/data.shape[0]*100
        df.loc[study_id, 'N3'] = sum(np.isin(data.Stage, [1]))/data.shape[0]*100
    except Exception as e:
        print(e)
        print(study_id)


  0%|          | 0/500 [00:00<?, ?it/s]/home/wolfgang/anaconda3/lib/python3.7/site-packages/ipykernel_launcher.py:8: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  
/home/wolfgang/anaconda3/lib/python3.7/site-packages/ipykernel_launcher.py:9: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  if __name__ == '__main__':

100%|██████████| 500/500 [47:47<00:00,  5.74s/it]


In [27]:
df.index.name = 'study_id'

In [28]:
df.to_csv('sleeplab_sleepindices_list.csv')

In [47]:
df.R.sort_values()[:5]

study_id
178    0.0
72     0.0
202    0.0
68     0.0
212    0.0
Name: R, dtype: float64

In [48]:
# AHI - low
# 55, 424
# AHI - high:
# 332, 317
# OI, CI phenotypes, same AHI
# 44, 79
# high CI:
# 269

# R-low:
# 178, 72
# R-high:
# 205, 58
# N3-high:
# 11, 267
# N3-low:
# 465, 389



In [ ]:
icu_micro_list['study_id'] = icu_micro_list['night_id'].apply(lambda x: x.split('_')[0])